<a href="https://colab.research.google.com/github/MattLScruggs/arclight-febe-dev/blob/main/notebooks/BPE_Tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Builds custom tokenizer using subset of epidemiological data

Import Libraries

In [1]:
from collections import Counter, deque
from functools import lru_cache
import re
import json
import os

Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/drive')

Mounted at /drive


Define BPE Class

In [3]:
class BPETokenizerSimple:
    def __init__(self):
        self.vocab = {}
        self.inverse_vocab = {}
        self.bpe_merges = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """
        tokens = self.pretokenize_text(text)

        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id


    def encode(self, text, allowed_special=None):
        """

        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.

        """

        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        token_ids = []
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"

            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally

                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()

            text = text[last_index:]

            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

        tokens = self.pretokenize_text(text)

        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))

        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Args:
            token (str): The token to tokenize.
        """
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")


        can_merge = True
        while can_merge and len(token_ids) > 1:
            can_merge = False
            new_tokens = []
            i = 0
            while i < len(token_ids) - 1:
                pair = (token_ids[i], token_ids[i + 1])
                if pair in self.bpe_merges:
                    merged_token_id = self.bpe_merges[pair]
                    new_tokens.append(merged_token_id)
                    i += 2  # Skip the next token as it's merged
                    can_merge = True
                else:
                    new_tokens.append(token_ids[i])
                    i += 1
            if i < len(token_ids):
                new_tokens.append(token_ids[i])
            token_ids = new_tokens
        return token_ids

        symbols = [self.vocab[id_num] for id_num in token_ids]

        while True:
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            if bigram is None or bigram not in self.bpe_ranks:
                break

            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Args:
            token_ids (List[int]): The list of token IDs to decode.

        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            if tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")
                        tokens.append("Ġ" + word)
                        pending_spaces = 0
                    else:
                        tokens.append(word)

            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequences = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    dq.popleft()
                else:
                    replaced.append(current)

            replaced_sequences.append(replaced)

        return replaced_sequences

### Unit Testing

Import testing JSONL

In [4]:
text_data = []
with open("/drive/MyDrive/Arclight/Datasets/JSONL_Output/test.jsonl", "r", encoding="utf-8") as file:
  for line in file:
    if line.strip():
      data=json.loads(line)
      text_data.append(data['text'])

In [5]:
flattened_text = "\n\n".join(text_data)

In [6]:
tokenizer=BPETokenizerSimple()

In [7]:
tokenizer.train(text=flattened_text, vocab_size=2000, allowed_special={"<|endoftext|>"})

In [8]:
tokenizer.save_vocab_and_merges(vocab_path="/drive/MyDrive/Arclight/Code/Utils/vocab.json",
                                bpe_merges_path="/drive/MyDrive/Arclight/Code/Utils/bpe_merges.txt")

Test encoding and decoding on short sample text

In [9]:
test_text= "Epidemiology is the scientific study of the distribution and determinants of health-related states or events in specified populations, and the application of this knowledge to control health problems. <|endoftext|> "

In [10]:
token_ids = tokenizer.encode(test_text, allowed_special= "<|endoftext|>")
print(token_ids)

[69, 112, 105, 100, 101, 109, 105, 111, 108, 111, 1984, 337, 256, 116, 104, 101, 538, 101, 110, 116, 105, 102, 309, 287, 116, 117, 100, 121, 256, 111, 102, 256, 116, 104, 101, 256, 100, 292, 116, 582, 98, 117, 116, 105, 111, 110, 282, 110, 100, 256, 100, 101, 116, 101, 114, 109, 105, 110, 97, 110, 116, 115, 256, 111, 102, 317, 101, 304, 116, 104, 45, 114, 101, 108, 97, 116, 101, 100, 287, 116, 97, 116, 101, 115, 256, 111, 114, 256, 101, 118, 101, 110, 116, 115, 256, 105, 110, 436, 101, 401, 102, 105, 101, 100, 297, 111, 112, 392, 97, 116, 105, 111, 110, 413, 282, 110, 100, 256, 116, 104, 101, 692, 815, 99, 97, 116, 105, 111, 110, 256, 111, 102, 256, 116, 104, 292, 474, 110, 111, 119, 108, 101, 100, 103, 101, 256, 116, 111, 288, 111, 110, 116, 114, 111, 108, 317, 101, 304, 116, 104, 903, 111, 98, 108, 101, 109, 390, 256, 60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62, 256]


In [11]:
print("Number of characters:", len(test_text))
print("Number of token IDs:", len(token_ids))
print("Compression ratio:", len(token_ids) / len(test_text))

Number of characters: 215
Number of token IDs: 182
Compression ratio: 0.8465116279069768


Test encoding and decoding on much longer text

In [12]:
long_text = []
with open("/drive/MyDrive/Arclight/Datasets/JSONL_Output/mmwr_archive.jsonl", "r", encoding="utf-8") as file:
  for line in file:
    if line.strip():
      data=json.loads(line)
      long_text.append(data['text_content'])

In [13]:
flattened_long_text = "\n\n".join([text for text in long_text if text is not None and isinstance(text, str)])

In [14]:
long_test_token_ids = tokenizer.encode(flattened_long_text)
print("Number of characters:", len(flattened_long_text))
print("Number of token IDs:", len(long_test_token_ids))
print("Compression ratio:", len(long_test_token_ids) / len(flattened_long_text))

Number of characters: 116101401
Number of token IDs: 52380154
Compression ratio: 0.4511586729259193


Testing saving and retrieval

In [15]:
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="/drive/MyDrive/Arclight/Code/Utils/vocab.json",
                                bpe_merges_path="/drive/MyDrive/Arclight/Code/Utils/bpe_merges.txt")

In [16]:
print(tokenizer2.decode(token_ids))

Epidemiology is the scientific study of the distribution and determinants of health-related states or events in specified populations, and the application of this knowledge to control health problems. <|endoftext|> 


Build full tokenizer and save, this time using HuggingFace's tokenizer library

In [31]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import ByteLevel as ByteLevelProcessor
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [18]:
tokenizer = Tokenizer(BPE(unk_token="<unk>"))

In [19]:
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

In [20]:
trainer = BpeTrainer(
    vocab_size=32768,
    special_tokens=[
        "<unk>",
        "<|endoftext|>",
    ],
    show_progress=True
)

In [23]:
tokenizer.train_from_iterator([flattened_long_text], trainer)

In [24]:
tokenizer.post_processor = ByteLevelProcessor(trim_offsets=True)

In [25]:
tokenizer.save("/drive/MyDrive/Arclight/Code/Utils/hf_tokenizer.json")

Testing the new tokenizer

In [27]:
token_ids_2=tokenizer.encode(test_text)
print(token_ids_2.ids)

[37, 1557, 1178, 285, 130, 9578, 1130, 120, 130, 3319, 122, 6268, 120, 341, 13, 1410, 992, 170, 2041, 117, 4909, 2168, 12, 122, 130, 6995, 120, 442, 4969, 147, 1360, 341, 4227, 14, 94, 1, 94]


In [32]:
tokenizer.decoder = ByteLevelDecoder()

In [33]:
decoded_text = tokenizer.decode(token_ids_2.ids)
print(decoded_text)

Epidemiology is the scientific study of the distribution and determinants of health-related states or events in specified populations, and the application of this knowledge to control health problems.  


Save notebook as script for later function importing

In [34]:
!jupyter nbconvert --to script "/drive/MyDrive/Arclight/Code/BPE_Tokenizer.ipynb"

[NbConvertApp] Converting notebook /drive/MyDrive/Arclight/Code/BPE_Tokenizer.ipynb to script
[NbConvertApp] Writing 14184 bytes to /drive/MyDrive/Arclight/Code/BPE_Tokenizer.txt


In [35]:
os.rename(
    "/drive/MyDrive/Arclight/Code/BPE_Tokenizer.txt",
    "/drive/MyDrive/Arclight/Code/BPE_Tokenizer.py"
)